In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


In [ ]:
pip install transformers

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")
for metin in ["gelcekmsn", "halısaya aldık", "b3dava", "Değerli müşterimiz kartınız bloke oldu [LINK] tıklayın"]:
    print(tok.tokenize(metin))
    

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("FacebookAI/xlm-roberta-base", num_labels=2)
model = model.to("cuda")

print(sum(p.numel() for p in model.parameters()) / 1e6, "milyon parametre")
print(torch.cuda.memory_allocated() / 1e9, "GB GPU belleği kullanılıyor")

In [ ]:
# === 08_zor_deneyler: XLM-R ile A/B/C/D/E, sınav = zor set (131 gerçek mesaj) ===

# --- 1. Kütüphaneler ---
import sys, time
sys.path.append("..")
import numpy as np
import pandas as pd
import torch
import mlflow
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
from sklearn.metrics import f1_score, precision_score, recall_score
hf_logging.set_verbosity_error()                  # uzun yükleme raporlarını gizle

# --- 2. Ayarlar: dünkü XLM-R ayarlarıyla aynı (max_len 160, uzun kurumsal mesajlar kesilmesin) ---
AYAR = {"model": "FacebookAI/xlm-roberta-base", "max_len": 160, "batch": 16,
        "lr": 2e-5, "epoch": 4, "warmup": 0.1, "weight_decay": 0.01, "seed": 42}
tok = AutoTokenizer.from_pretrained(AYAR["model"])

# --- 3. MLflow: zor set deneyleri ayrı bir deney grubunda dursun ---
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("tr_zor")

# --- 4. Veri yükleyici: mesajları 16'lık gruplara böl, her grubu kendi içinde doldur ---
def yukleyici(metinler, etiketler, a, karistir):
    veri = list(zip(metinler, etiketler))
    def birlestir(parti):
        m, e = zip(*parti)
        kod = tok(list(m), padding=True, truncation=True, max_length=a["max_len"], return_tensors="pt")
        kod["labels"] = torch.tensor(e)
        return kod
    return DataLoader(veri, batch_size=a["batch"], shuffle=karistir, collate_fn=birlestir)

# --- 5. Eğit + tahmin: her çağrıda TAZE bir XLM-R ---
def egit_ve_tahmin(X_e, y_e, X_d, a):
    torch.manual_seed(a["seed"])
    m = AutoModelForSequenceClassification.from_pretrained(a["model"], num_labels=2).to("cuda")
    egitim = yukleyici(X_e, y_e, a, karistir=True)
    opt = torch.optim.AdamW(m.parameters(), lr=a["lr"], weight_decay=a["weight_decay"])
    toplam = len(egitim) * a["epoch"]
    plan = get_linear_schedule_with_warmup(opt, int(toplam * a["warmup"]), toplam)

    # 5a. eğitim döngüsü
    m.train()
    for _ in range(a["epoch"]):
        for parti in egitim:
            parti = {k: v.to("cuda") for k, v in parti.items()}
            with torch.autocast("cuda", dtype=torch.bfloat16):      # yarım hassasiyet: hızlı + az bellek
                kayip = m(**parti).loss
            kayip.backward(); opt.step(); plan.step(); opt.zero_grad()

    # 5b. tahmin: spam olasılığı
    m.eval(); olasilik = []
    with torch.no_grad():
        for parti in yukleyici(X_d, [0] * len(X_d), a, karistir=False):
            parti = {k: v.to("cuda") for k, v in parti.items() if k != "labels"}
            with torch.autocast("cuda", dtype=torch.bfloat16):
                logit = m(**parti).logits
            olasilik.append(torch.softmax(logit.float(), dim=-1)[:, 1].cpu().numpy())

    # 5c. GPU belleğini boşalt (sonraki model için)
    del m, opt
    torch.cuda.empty_cache()
    return np.concatenate(olasilik)

# --- 6. Ölçüm: Esra'nın tablosundaki sayılarla aynı ---
def zor_olc(y, p, etiket):
    t = (p > 0.5).astype(int)
    dol = etiket == "dolandırıcılık"
    return {"zor_f1": f1_score(y, t), "zor_p": precision_score(y, t), "zor_r": recall_score(y, t),
            "yanlis_alarm": int(t[y == 0].sum()), "spam_yakalama": int(t[y == 1].sum()),
            "dol_yakalama": int(t[dol].sum())}

print("kurulum tamam,", torch.cuda.get_device_name(0))

In [ ]:
# --- 1. Eğitim setini kur: kaynakları birleştir, sınav göndericisini çıkar ---
def egitim_seti(kaynaklar, haric_grup=None):
    X, y = [], []
    for m, yy, g in kaynaklar:
        tut = (g != haric_grup) if haric_grup is not None else np.ones(len(m), bool)
        X += [m[i] for i in np.where(tut)[0]]        # sınav göndericisinin mesajlarını atla
        y.append(yy[tut])
    return X, np.concatenate(y)

# --- 2. Tek bir deney: eğit, zor sette ölç, MLflow'a yaz ---
def deney(ad, ekler, loso):
    """ekler: {"S2": S2, ...}  loso: True ise zor setin diğer göndericileri de eğitime girer"""
    basla = time.time()
    p = np.zeros(len(Z[0]))
    ek_kaynaklar = list(ekler.values())

    if not loso:
        # 2a. Tek eğitim: T + ekler, sınav = zor setin tamamı
        X_e, y_e = egitim_seti([T] + ek_kaynaklar)
        p = egit_ve_tahmin(X_e, y_e, Z[0], AYAR)
    else:
        # 2b. Leave-one-sender-out: her turda bir gönderici sınav, gerisi eğitimde
        gruplar = np.unique(Z[2])
        for i, g in enumerate(gruplar):
            sinav = Z[2] == g
            X_e, y_e = egitim_seti([T, Z] + ek_kaynaklar, haric_grup=g)   # Z ve SAL'dan g çıkarılır
            p[sinav] = egit_ve_tahmin(X_e, y_e, [Z[0][j] for j in np.where(sinav)[0]], AYAR)
            if (i + 1) % 10 == 0:
                print(f"  {ad}: {i + 1}/{len(gruplar)}, {(time.time() - basla) / 60:.1f} dk")

    # 2c. Ölç ve kaydet
    sonuc = zor_olc(Z[1], p, zor["etiket"].to_numpy())
    with mlflow.start_run(run_name=f"xlmr_zor_{ad}"):
        mlflow.log_params({**AYAR, "deney": ad, "ekler": "+".join(ekler) or "yok",
                           "loso": loso, "sinav": "zor_gercek v2 (131)"})
        mlflow.log_metrics(sonuc)
    print(f"{ad} bitti, {(time.time() - basla) / 60:.1f} dk")
    return p, sonuc

# --- 3. Hızlı deneyler: tek eğitim, her biri ~20-30 sn ---
tahmin, sonuc = {}, {}
for ad, ekler in [("A", {}), ("C1", {"S1": S1}), ("C2", {"S2": S2})]:
    tahmin[ad], sonuc[ad] = deney(ad, ekler, loso=False)

# --- 4. Tablo ---
pd.DataFrame(sonuc).T.round(3)

In [ ]:
# --- LOSO deneyleri: her biri 58 eğitim, tahminen 15-25 dk. Toplam ~1 saat ---
for ad, ekler in [("B", {}),                          # + gerçek kurumsal (diğer göndericiler)
                  ("D", {"S2": S2}),                  # + gerçek + iyi sentetik
                  ("E", {"S2": S2, "SAL": SAL})]:     # + gerçek + iyi sentetik + senin 58 mesajın
    tahmin[ad], sonuc[ad] = deney(ad, ekler, loso=True)

# --- Tüm deneyler tek tabloda ---
pd.DataFrame(sonuc).T.round(3)

In [ ]:
# --- G: tr_train + sentetik v2/v3 + havuz (3 versiyon). Esra'nın zor setinden HİÇBİR ŞEY yok ---
for v in ["tum", "sablon", "40"]:
    # 1. Havuz versiyonunu oku
    hv = pd.read_csv(D + f"salih_havuz_{v}.csv", dtype=str, keep_default_na=False)
    HVv = hazirla(hv, "gonderen")

    # 2. Zor setle ortak göndericileri çıkar (G'de LOSO yok, ortak gönderici = şablon sızıntısı)
    temiz = ~np.isin(HVv[2], np.unique(Z[2]))
    HV_temiz = ([HVv[0][i] for i in np.where(temiz)[0]], HVv[1][temiz], HVv[2][temiz])
    print(f"HV_{v}: {len(HVv[0])} → ortak göndericiler çıkınca {len(HV_temiz[0])}")

    # 3. Deney
    tahmin[f"G_{v}"], sonuc[f"G_{v}"] = deney(f"G_{v}", {"S2": S2, f"HV_{v}": HV_temiz}, loso=False)

# --- Bütün deneyler ---
pd.DataFrame(sonuc).T.round(3)

In [ ]:
# --- G_retsiz: havuzdan ret/iptal kısmını kes, model kuralı öğrenemesin ---
import re

# --- 1. Kural (09'daki RET ile aynı; 07'nin belleğinde yok, burada tekrar tanımlıyoruz) ---
NO = r"(?:\d{4}|\[TEL\])"
RET = re.compile(
    r"sms\s*(?:iptal|ipt|ret|almak\s+istemiyor|almamak)|\bRET\b|IZINYOK|IPTALIZIN|\bcks\s*:"
    r"|yaz(?:ıp|ip|ın|in|arak)?\s*" + NO +
    r"|" + NO + r"\s*['’]?\s*(?:e|a|ye|ya)\s+(?:gönder|gonder|ilet)"
    r"|(?:iptal|ret)\w*\s+yaz", re.IGNORECASE)

# --- 2. Ret ifadesinin başladığı yerden mesajın sonuna kadar kes ---
def ret_kes(metin):
    m = RET.search(metin)
    return metin[:m.start()].rstrip(" .,:;-") if m else metin

for v in ["sablon", "40"]:
    hv = pd.read_csv(D + f"salih_havuz_{v}.csv", dtype=str, keep_default_na=False)
    hv["mesaj"] = hv["mesaj"].apply(ret_kes)
    hv = hv[hv["mesaj"].str.len() >= 15]                     # kesince boş kalanları at

    # --- 3. Kontrol: kesilmiş havuzda hâlâ ret ifadesi kalan var mı? (0 olmalı) ---
    print(f"HV_{v}_retsiz: {len(hv)} mesaj, ret kalan: {hv['mesaj'].str.contains(RET).sum()}")

    # --- 4. G ile aynı: ortak göndericileri çıkar, deneyi koş ---
    HVv = hazirla(hv, "gonderen")
    temiz = ~np.isin(HVv[2], np.unique(Z[2]))
    HV_temiz = ([HVv[0][i] for i in np.where(temiz)[0]], HVv[1][temiz], HVv[2][temiz])
    ad = f"G_{v}_retsiz"
    tahmin[ad], sonuc[ad] = deney(ad, {"S2": S2, f"HV_{v}_retsiz": HV_temiz}, loso=False)

pd.DataFrame(sonuc).T.round(3)

In [ ]:
# --- F: E + havuz (şablon, retsiz). LOSO: her turda sınav göndericisi havuzdan da çıkar ---
hv = pd.read_csv(D + "salih_havuz_sablon.csv", dtype=str, keep_default_na=False)
hv["mesaj"] = hv["mesaj"].apply(ret_kes)
hv = hv[hv["mesaj"].str.len() >= 15]
HV_F = hazirla(hv, "gonderen")                 # ortak göndericileri ÇIKARMIYORUZ, LOSO her turda çıkarır

tahmin["F"], sonuc["F"] = deney("F", {"S2": S2, "SAL": SAL, "HV_sablon_retsiz": HV_F}, loso=True)
pd.DataFrame(sonuc).T.round(3)

> Not: Bu sayılar etiket düzeltmesinden önceki zor setle (v1) hesaplandı. Güncel sonuçlar: Gün 8.

In [ ]:
import time
import numpy as np
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score

del model
torch.cuda.empty_cache()

AYAR = {"model": "FacebookAI/xlm-roberta-base", "max_len": 80, "batch": 16,
        "lr": 2e-5, "epoch": 4, "warmup": 0.1, "weight_decay": 0.01, "seed": 42}

X = train["mesaj"].tolist()
y = (train["ikili"] == "spam").astype(int).to_numpy()
katlar = train["fold"].to_numpy()

def yukleyici(metinler, etiketler, a, karistir):
    veri = list(zip(metinler, etiketler))
    def birlestir(parti):
        m, e = zip(*parti)
        kod = tok(list(m), padding=True, truncation=True, max_length=a["max_len"], return_tensors="pt")
        kod["labels"] = torch.tensor(e)
        return kod
    return DataLoader(veri, batch_size=a["batch"], shuffle=karistir, collate_fn=birlestir)

def egit_ve_tahmin(X_e, y_e, X_d, a):
    torch.manual_seed(a["seed"])
    m = AutoModelForSequenceClassification.from_pretrained(a["model"], num_labels=2).to("cuda")
    egitim = yukleyici(X_e, y_e, a, karistir=True)
    opt = torch.optim.AdamW(m.parameters(), lr=a["lr"], weight_decay=a["weight_decay"])
    toplam = len(egitim) * a["epoch"]
    plan = get_linear_schedule_with_warmup(opt, int(toplam * a["warmup"]), toplam)

    m.train()
    for ep in range(a["epoch"]):
        for parti in egitim:
            parti = {k: v.to("cuda") for k, v in parti.items()}
            with torch.autocast("cuda", dtype=torch.bfloat16):
                kayip = m(**parti).loss
            kayip.backward()
            opt.step()
            plan.step()
            opt.zero_grad()

    m.eval()
    olasilik = []
    with torch.no_grad():
        for parti in yukleyici(X_d, [0] * len(X_d), a, karistir=False):
            parti = {k: v.to("cuda") for k, v in parti.items() if k != "labels"}
            with torch.autocast("cuda", dtype=torch.bfloat16):
                logit = m(**parti).logits
            olasilik.append(torch.softmax(logit.float(), dim=-1)[:, 1].cpu().numpy())

    del m, opt
    torch.cuda.empty_cache()
    return np.concatenate(olasilik)

basla = time.time()
egit, dogr = katlar != 0, katlar == 0
p = egit_ve_tahmin([X[i] for i in np.where(egit)[0]], y[egit], [X[i] for i in np.where(dogr)[0]], AYAR)
print(f"fold 0 F1: {f1_score(y[dogr], (p > 0.5).astype(int)):.3f}")
print(f"süre: {time.time() - basla:.0f} sn, en yüksek GPU belleği: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

In [ ]:
import mlflow
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("tr_ikili")

def xlmr_cv(a):
    skorlar = []
    for k in range(5):
        egit, dogr = katlar != k, katlar == k
        p = egit_ve_tahmin([X[i] for i in np.where(egit)[0]], y[egit],
                           [X[i] for i in np.where(dogr)[0]], a)
        skorlar.append(f1_score(y[dogr], (p > 0.5).astype(int)))
        print(f"fold {k}: {skorlar[-1]:.3f}")
    return np.array(skorlar)

with mlflow.start_run(run_name="xlmr_base"):
    skor = xlmr_cv(AYAR)
    mlflow.log_params(AYAR)
    mlflow.log_metrics({"cv_f1": skor.mean(), "cv_f1_std": skor.std()})
    print(skor.mean().round(3), skor.round(3))

In [ ]:
oof = np.zeros(len(y))
for k in range(5):
    egit, dogr = katlar != k, katlar == k
    oof[dogr] = egit_ve_tahmin([X[i] for i in np.where(egit)[0]], y[egit],
                               [X[i] for i in np.where(dogr)[0]], AYAR)

train["xlmr_olasilik"] = oof
hatalar = train[(oof > 0.5).astype(int) != y]
print(len(hatalar), "hata,", (hatalar["fold"] == 2).sum(), "tanesi fold 2'de")
hatalar[["id", "fold", "ikili", "xlmr_olasilik", "mesaj"]].sort_values("fold")

In [ ]:
zor = pd.read_csv("C:\\Users\\SCT\\Projects\\sms-fraud-detection\\external\\zor_gercek_hepsi.csv")
zor = zor = zor[zor["etiket"].notna()].reset_index(drop=True)
zor["y"] = (zor["etiket"] != "meşru").astype(int)

print(zor.shape, zor["y"].value_counts().to_dict(), zor["gonderen"].nunique(), "gönderici")
uzun = zor["mesaj"].apply(lambda m: len(tok(m)["input_ids"]))
print("80'i aşan:", (uzun > 80).sum(), " en uzun:", uzun.max())

In [ ]:
AYAR_Z = {**AYAR, "max_len": 160}

def zor_olc(y_gercek, olasilik):
    t = (olasilik > 0.5).astype(int)
    meşru, spam = y_gercek == 0, y_gercek == 1
    return {
        "zor_f1": f1_score(y_gercek, t),
        "zor_p": precision_score(y_gercek, t),
        "zor_r": recall_score(y_gercek, t),
        "yanlis_alarm": int(t[meşru].sum()),
        "spam_yakalama": int(t[spam].sum()),
    }

from sklearn.metrics import precision_score, recall_score

basla = time.time()
p_A = egit_ve_tahmin(X, y, zor["mesaj"].tolist(), AYAR_Z)
sonuc_A = zor_olc(zor["y"].to_numpy(), p_A)

with mlflow.start_run(run_name="xlmr_zor_A"):
    mlflow.log_params({**AYAR_Z, "egitim": "tr_train", "sinav": "zor_gercek_133"})
    mlflow.log_metrics(sonuc_A)

print(sonuc_A)
print(f"yanlış alarm {sonuc_A['yanlis_alarm']}/68, spam yakalama {sonuc_A['spam_yakalama']}/65, "
      f"süre {time.time() - basla:.0f} sn, GPU {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

In [ ]:
zor["grup"] = zor["gonderen"].str.upper().str.strip()
gruplar = zor["grup"].unique()
print(len(gruplar), "gönderici grubu")

p_B = np.zeros(len(zor))
basla = time.time()
for i, g in enumerate(gruplar):
    sinav = (zor["grup"] == g).to_numpy()
    ek = zor[~sinav]
    X_e = X + ek["mesaj"].tolist()
    y_e = np.concatenate([y, ek["y"].to_numpy()])
    p_B[sinav] = egit_ve_tahmin(X_e, y_e, zor.loc[sinav, "mesaj"].tolist(), AYAR_Z)
    if (i + 1) % 10 == 0:
        print(f"{i + 1}/{len(gruplar)} bitti, {(time.time() - basla) / 60:.1f} dk")

sonuc_B = zor_olc(zor["y"].to_numpy(), p_B)
with mlflow.start_run(run_name="xlmr_zor_B"):
    mlflow.log_params({**AYAR_Z, "egitim": "tr_train + diğer göndericiler", "sinav": "leave-one-sender-out (büyük harf)"})
    mlflow.log_metrics(sonuc_B)

zor["p_A"], zor["p_B"] = p_A, p_B
print(sonuc_B)
print(f"yanlış alarm {sonuc_B['yanlis_alarm']}/68, spam yakalama {sonuc_B['spam_yakalama']}/65, "
      f"GPU {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

In [ ]:
# --- 1. Tahminleri etikete çevir ---
zor["t_A"] = (zor["p_A"] > 0.5).astype(int)
zor["t_B"] = (zor["p_B"] > 0.5).astype(int)

# --- 2. Hangi mesajlar A'da yanlış, B'de doğru oldu? (düzelenler) ---
duzelen = zor[(zor["t_A"] != zor["y"]) & (zor["t_B"] == zor["y"])]
print("Düzelen:", len(duzelen))

# --- 3. B'de hâlâ yanlış olanlar ---
kalan = zor[zor["t_B"] != zor["y"]].copy()
kalan["hata"] = np.where(kalan["y"] == 0, "yanlış alarm", "kaçan spam")   # hata türü etiketi
print("B'de hâlâ yanlış:", len(kalan))

# --- 4. Dolandırıcılık özelinde: 11 gerçek dolandırıcılığın kaçı yakalandı? ---
dol = zor["etiket"] == "dolandırıcılık"
print(f"Dolandırıcılık yakalama  A: {zor.loc[dol, 't_A'].sum()}/{dol.sum()}  "
      f"B: {zor.loc[dol, 't_B'].sum()}/{dol.sum()}")

# --- 5. Kalan hataları göster (kısa mesaj önizlemesiyle) ---
kalan["onizleme"] = kalan["mesaj"].str[:70]          # ilk 70 karakter, tablo taşmasın
kalan.sort_values("hata")[["hata", "grup", "etiket", "p_A", "p_B", "onizleme"]].round(2)

In [ ]:
# --- 1. Paylaşılacak sütunları seç ---
cikti = zor[["gonderen", "grup", "etiket", "y", "p_A", "p_B", "t_A", "t_B", "mesaj"]]

# --- 2. Gitignore'daki klasöre yaz (repoya girmez) ---
cikti.to_csv("C:\\Users\\SCT\\Projects\\sms-fraud-detection\\external\\zor_gercek_hepsi.csv", index=False, encoding="utf-8-sig")

# --- 3. Dolandırıcılık özetini tekrar yazdır (önceki çıktıda görünmemişti) ---
dol = zor["etiket"] == "dolandırıcılık"
print(f"Dolandırıcılık yakalama  A: {zor.loc[dol, 't_A'].sum()}/{dol.sum()}  "
      f"B: {zor.loc[dol, 't_B'].sum()}/{dol.sum()}")

In [ ]:
# --- Zor seti oku (göreli yol: notebooks/ klasöründen bir üst klasöre çık) ---
zor = pd.read_csv("../data/external/zor_gercek_hepsi.csv")